In [ ]:
# ============================================================
# DEEP_DIVE_YUKAWA_TORCH_V4: ЮКАВА НА GPU С ПОДРОБНЫМ REM
# ============================================================

# REM: Импортируем библиотеку numpy для работы с массивами на CPU.
import numpy as np

# REM: Импортируем PyTorch. Это библиотека для тензорных вычислений на GPU.
import torch

# REM: Импортируем модуль functional из PyTorch.
# REM: Он нужен для операции свертки (conv2d), которая будет считать периметр на GPU.
import torch.nn.functional as F

# REM: Импортируем функцию label из scipy.
# REM: Она понадобится для поиска кластеров (P_inf) на CPU.
from scipy.ndimage import label

# REM: Импортируем matplotlib для отрисовки графиков.
import matplotlib

# REM: Используем фоновый режим 'Agg', чтобы графики не открывались в окнах.
matplotlib.use('Agg') 

# REM: Импортируем pyplot для создания картинок.
import matplotlib.pyplot as plt

# REM: Импортируем csv для записи таблиц с результатами.
import csv

# REM: Импортируем time для замера времени выполнения расчетов.
import time

# REM: Импортируем os для работы с файловой системой (создание папок).
import os

# REM: Импортируем datetime для создания метки времени в названии папки.
from datetime import datetime

# REM: Импортируем функции для вывода картинок прямо в блокнот Jupyter.
from IPython.display import display, Image as IPImage

# ------------------------------------------------------------
# REM: БЛОК 1. НАСТРОЙКА УСТРОЙСТВА И ПАПОК
# ------------------------------------------------------------

# REM: Проверяем, доступна ли CUDA (видеокарта).
# REM: Если доступна, device будет 'cuda'. Если нет — 'cpu' (программа не упадет, но будет медленной).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# REM: Выводим в консоль, какое устройство используется.
print(f"Устройство: {device}", flush=True)

# REM: Генерируем метку времени (например, 20261029_153045).
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# REM: Создаем имя главной папки для результатов с приклеенной меткой времени.
output_dir = f"DEEP_DIVE_YUKAWA_GPU_{timestamp}"

# REM: Создаем пути к подпапкам для CSV, PNG и чекпоинтов.
csv_dir = os.path.join(output_dir, "csv")
png_dir = os.path.join(output_dir, "png")
ckpt_dir = os.path.join(output_dir, "checkpoints")

# REM: Создаем все эти папки. exist_ok=True не даст упасть, если они уже есть.
os.makedirs(output_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(png_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

# ------------------------------------------------------------
# REM: БЛОК 2. ФИЗИЧЕСКИЕ ПАРАМЕТРЫ
# ------------------------------------------------------------

# REM: L - размер решетки (100x100 узлов).
L = 100

# REM: STEPS - число шагов внешнего цикла.
# REM: В коде умножение на 5, поэтому реальных шагов будет 500 000 * 5 = 2 500 000.
STEPS = 500_000

# REM: beta - обратная температура (1/kT). Отвечает за тепловой шум.
beta = 1.0

# REM: J0 - сила притяжения Юкавы.
J0 = 3.0

# REM: mu - параметр экранирования (как быстро затухает сила).
mu = 0.5

# REM: r_cutoff - радиус обрезания (считать взаимодействие в радиусе 4 клеток).
r_cutoff = 4

# REM: Список точек для расчета: (Плотность, Сигма).
POINTS = [
    (0.10, 2.0),   
    (0.10, 10.0),  
    (0.50, 1.5),   
    (0.50, 2.0)    
]

# REM: Формируем путь к CSV-файлу для записи результатов.
csv_file = os.path.join(csv_dir, "deep_dive_yukawa_gpu.csv")

# REM: Если файла нет, создаем его и пишем заголовки.
if not os.path.exists(csv_file):
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['p_init', 'J0', 'sigma', 'step', 'density', 'P_inf', 'perimeter', 'max_size'])

# ------------------------------------------------------------
# REM: БЛОК 3. ЯДРА ДЛЯ СВЕРТОК
# ------------------------------------------------------------

def make_yukawa_kernel(r_cut, mu, J0_val):
    # REM: Функция для создания 2D-матрицы (ядра) сил Юкавы.
    # REM: Создаем нулевую матрицу размера 9x9 (так как r_cut=4, то 2*4+1=9).
    size = 2 * r_cut + 1
    kernel = np.zeros((1, 1, size, size), dtype=np.float32)
    
    # REM: Заполняем ядро по формуле Юкавы.
    for i in range(size):
        for j in range(size):
            dx, dy = i - r_cut, j - r_cut
            r = np.sqrt(dx**2 + dy**2)
            if r > 0 and r <= r_cut:
                kernel[0, 0, i, j] = J0_val * np.exp(-mu * r) / r
    
    # REM: Возвращаем ядро как тензор PyTorch на GPU.
    return torch.tensor(kernel, device=device)

# REM: Создаем ядро Юкавы для текущих параметров.
yukawa_kernel = make_yukawa_kernel(r_cutoff, mu, J0)

def calculate_perimeter_torch(grid):
    # REM: Создаем ядро 3x3 для подсчета периметра.
    # REM: Единицы стоят по бокам (сверху, снизу, слева, справа).
    perim_kernel = torch.tensor([
        [0, 1, 0],
        [1, 0, 1],
        [0, 1, 0]
    ], dtype=torch.float32, device=device).view(1, 1, 3, 3)
    
    # REM: Преобразуем сетку в тензор формата (1 канал, 1 слой, L, L).
    grid_f = grid.float().view(1, 1, L, L)
    
    # REM: Делаем паддинг (рамку) с зацикливанием (circular), чтобы учесть границы.
    padded = F.pad(grid_f, (1, 1, 1, 1), mode='circular')
    
    # REM: Выполняем свертку. Она умножает каждую клетку на ядро и суммирует.
    # REM: Это дает количество границ (где 1 встречается с 0).
    perim = F.conv2d(padded, perim_kernel).sum()
    
    # REM: Возвращаем полученный периметр (тензор).
    return perim

# ------------------------------------------------------------
# REM: БЛОК 4. ФИЗИЧЕСКИЙ ШАГ МЕТРОПОЛИСА
# ------------------------------------------------------------

def metropolis_step_torch(grid, sigma, kernel, beta=1.0):
    # REM: Генерируем случайные координаты (i1, j1) на CPU и переводим в список.
    i1, j1 = torch.randint(0, L, (2,)).tolist()
    
    # REM: Ищем занятую клетку (1). Пока не найдем — генерируем новые координаты.
    while grid[i1, j1] != 1:
        i1, j1 = torch.randint(0, L, (2,)).tolist()
        
    # REM: Генерируем случайные координаты (i2, j2) для пустой клетки.
    i2, j2 = torch.randint(0, L, (2,)).tolist()
    
    # REM: Ищем пустую клетку (0). Пока не найдем — генерируем новые.
    while grid[i2, j2] != 0:
        i2, j2 = torch.randint(0, L, (2,)).tolist()

    # REM: Создаем сетку смещений для окна вокруг узла (от -4 до 4).
    dy, dx = torch.meshgrid(torch.arange(-r_cutoff, r_cutoff+1, device=device),
                            torch.arange(-r_cutoff, r_cutoff+1, device=device), indexing='ij')
    
    # REM: Внутренняя функция для расчета локальной энергии одного узла.
    def get_energy(i, j):
        # REM: Вычисляем индексы окна с учетом зацикливания (остаток от деления).
        y_idx = (i + dy) % L
        x_idx = (j + dx) % L
        
        # REM: Вырезаем кусок сетки (патч) вокруг узла.
        patch = grid[y_idx, x_idx]
        
        # REM: Считаем энергию: умножаем патч на ядро Юкавы и суммируем со знаком минус.
        return -torch.sum(patch * kernel.squeeze())

    # REM: Считаем энергию ДО обмена для первого узла.
    E_old = get_energy(i1, j1)
    
    # REM: Считаем периметр ДО обмена (векторизовано).
    P_old = calculate_perimeter_torch(grid)

    # REM: Меняем узлы местами (1 -> 0, 0 -> 1).
    grid[i1, j1] = 0
    grid[i2, j2] = 1

    # REM: Считаем энергию ПОСЛЕ обмена для второго узла.
    E_new = get_energy(i2, j2)
    
    # REM: Считаем периметр ПОСЛЕ обмена.
    P_new = calculate_perimeter_torch(grid)

    # REM: Полное изменение энергии: (Энергия Юкавы) + Сигма * (Изменение периметра).
    delta_S = (E_new - E_old) + sigma * (P_new - P_old)

    # REM: Критерий Метрополиса.
    # REM: Если энергия упала (delta_S <= 0) — принимаем.
    # REM: Или если случайное число меньше exp(-beta * delta_S) — принимаем.
    if delta_S <= 0 or torch.rand(1).item() < np.exp(-beta * delta_S.item()):
        return True
    else:
        # REM: Если не приняли — возвращаем как было.
        grid[i1, j1] = 1
        grid[i2, j2] = 0
        return False

# ------------------------------------------------------------
# REM: БЛОК 5. ИЗМЕРЕНИЯ
# ------------------------------------------------------------

def measure_torch(grid):
    # REM: Переносим сетку с GPU на CPU в виде обычного numpy-массива.
    grid_cpu = grid.cpu().numpy()
    
    # REM: Добавляем периодические границы для корректного поиска кластеров.
    padded = np.pad(grid_cpu, pad_width=1, mode='wrap')
    
    # REM: Ищем кластеры 4-связности.
    labeled, n = label(padded, structure=[[0,1,0],[1,1,1],[0,1,0]])
    
    # REM: Убираем добавленную рамку.
    labeled = labeled[1:-1, 1:-1]
    
    # REM: Считаем размеры кластеров.
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    sizes = sizes[sizes > 0]
    
    # REM: Если кластеров нет, возвращаем None.
    if len(sizes) == 0: return None
    
    # REM: P_inf - доля самого большого кластера.
    P_inf = float(sizes.max() / np.sum(sizes))
    
    # REM: Реальная плотность.
    density = float(np.mean(grid_cpu))
    
    # REM: Периметр. Вызываем нашу функцию на GPU и забираем число.
    perimeter = int(calculate_perimeter_torch(grid).item())
    
    # REM: Максимальный размер.
    max_size = int(sizes.max())
    
    return {'density': density, 'P_inf': P_inf, 'perimeter': perimeter, 'max_size': max_size}

# ------------------------------------------------------------
# REM: БЛОК 6. ГЛАВНЫЙ ЦИКЛ
# ------------------------------------------------------------

for p, sigma in POINTS:
    # REM: Формируем пути к файлам чекпоинтов.
    ckpt_grid_file = os.path.join(ckpt_dir, f"grid_p{p}_s{sigma}.pt")
    ckpt_step_file = os.path.join(ckpt_dir, f"step_p{p}_s{sigma}.npy")
    
    start_step = 0
    grid = None
    
    # REM: Если есть сохранения — восстанавливаем.
    if os.path.exists(ckpt_grid_file) and os.path.exists(ckpt_step_file):
        grid = torch.load(ckpt_grid_file, weights_only=True).to(device)
        start_step = int(np.load(ckpt_step_file)[0])
        print(f"\n[RESTART] D {p*100}%, S {sigma}. Восстанавливаем...", flush=True)
    else:
        # REM: Если нет — создаем случайную сетку на GPU.
        grid = torch.tensor((np.random.random((L, L)) < p).astype(np.int8), device=device)

    print(f"\nСЧИТАЕМ GPU: D={p*100}%, Sigma={sigma}", flush=True)
    print(f"{'Шаг':>8} | {'density':>8} | {'P_inf':>8} | {'Perimeter':>10} | {'Time(s)':>7}", flush=True)
    print("-" * 55, flush=True)

    t0 = time.time()
    
    # REM: Основной цикл.
    for step in range(start_step, STEPS):
        # REM: Внутренний цикл (5 шагов за раз).
        for _ in range(5):
            metropolis_step_torch(grid, sigma, yukawa_kernel, beta)
        
        # REM: Реальный номер шага (с учетом умножения на 5).
        real_step = step * 5
        
        # REM: Каждые 250 000 шагов сохраняем.
        if real_step % 250_000 == 0 and real_step > 0:
            m = measure_torch(grid)
            elapsed = time.time() - t0
            
            with open(csv_file, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([p, J0, sigma, real_step, m['density'], m['P_inf'], m['perimeter'], m['max_size']])
            
            print(f"{real_step:>8} | {m['density']:>8.3f} | {m['P_inf']:>8.3f} | {m['perimeter']:>10.1f} | {elapsed:>7.1f}", flush=True)
            
            # REM: Сохраняем сетку и шаг.
            torch.save(grid, ckpt_grid_file)
            np.save(ckpt_step_file, np.array([step]))
            
            # REM: Создаем и сохраняем картинку.
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(grid.cpu().numpy(), cmap='binary', interpolation='nearest')
            ax.set_title(f'GPU Y D={p*100}%, σ={sigma}, Step={real_step}')
            img_path = os.path.join(png_dir, f"GPU_D{p}_S{sigma}_step{real_step}.png")
            plt.savefig(img_path, dpi=100, bbox_inches='tight')
            plt.close(fig)
            display(IPImage(filename=img_path))

    # REM: Финальный замер и запись.
    m = measure_torch(grid)
    with open(csv_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([p, J0, sigma, STEPS*5, m['density'], m['P_inf'], m['perimeter'], m['max_size']])
    
    # REM: Удаляем чекпоинты после успешного завершения.
    if os.path.exists(ckpt_grid_file): os.remove(ckpt_grid_file)
    if os.path.exists(ckpt_step_file): os.remove(ckpt_step_file)
    
    print(f"ТОЧКА ЗАВЕРШЕНА.", flush=True)

print(f"\nВСЕ 4 ТОЧКИ ГОТОВЫ! Данные в {output_dir}.", flush=True)